# Strategy - Tail Risk Timing

In [ ]:
# ============================================================
# S09 - Tail Risk Timing
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

def strategy(data):

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    close = xr.where(
        is_liquid == 1,
        close,
        np.nan
    )

    # --------------------------------------------------------
    # BTC Regime Filter
    # --------------------------------------------------------

    btc = data.sel(
        field="close",
        asset="BTC"
    )

    btc_sma200 = qnta.sma(
        btc,
        200
    )

    bull_market = xr.where(
        btc > btc_sma200,
        1,
        0
    )

    # --------------------------------------------------------
    # Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    vol60 = qnta.std(
        ret,
        60
    )

    # --------------------------------------------------------
    # Tail Score
    # --------------------------------------------------------

    tail_score = (
        ret /
        (vol60 + 1e-6)
    )

    score = tail_score.rank(
        "asset"
    )

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # Top Assets
    # --------------------------------------------------------

    ranks = score.rank(
        "asset"
    )

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Vol Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (
        vol20 + 1e-6
    )

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    denom = weights.sum(
        "asset"
    )

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    # --------------------------------------------------------
    # BTC Regime Filter
    # --------------------------------------------------------

    weights = weights * bull_market

    return weights.fillna(0)

weights = strategy(data)

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field          equity  relative_return  volatility  underwater  max_drawdown  \
time                                                                           
2026-06-05  51.120366              0.0    0.693118   -0.923035      -0.94002   
2026-06-06  51.120366              0.0    0.693027   -0.923035      -0.94002   
2026-06-07  51.120366              0.0    0.692936   -0.923035      -0.94002   
2026-06-08  51.120366              0.0    0.692845   -0.923035      -0.94002   
2026-06-09  51.120366              0.0    0.692755   -0.923035      -0.94002   

field       sharpe_ratio  mean_return  bias  instruments  avg_turnover  \
time                                                                     
2026-06-05      0.888171     0.615607   0.0         32.0      0.730972   
2026-06-06      0.888054     0.615445   0.0         32.0      0.730780   
2026-06-07      0.887937     0.615284   0.0         32.0      0.730588   
2026-06-08      0.887820     0.615122   0.0         32.0      0.730396   
2026-06-09      0.887704     0.614961   0.0         32.0      0.730205   

field       avg_holding_time  
time                          
2026-06-05          1.705103  
2026-06-06          1.705103  
2026-06-07          1.705103  
2026-06-08          1.705103  
2026-06-09          1.705103  

Final Metrics:
field
equity              51.120366
sharpe_ratio         0.887704
max_drawdown        -0.940020
avg_turnover         0.730205
avg_holding_time     1.705103
Name: 2026-06-09 00:00:00, dtype: float64